# Person 2: Character TF-IDF + Linear SVM
Run cells in order from the repository root or `code/`. Train-only fitting; validation is used for selection and the small sample set is supporting evidence. Set the real group ID before generating a submission.

## 1.Dependencies

In [1]:
from pathlib import Path
import time, json, platform
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from pythainlp.tokenize import word_tokenize
import pythainlp
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'code/data/train.csv').is_file()), None)
if ROOT is None: raise FileNotFoundError('Open from repository root or code folder')
DATA = ROOT / 'code/data'
train, valid, sample = [pd.read_csv(DATA / f) for f in ('train.csv','valid.csv','sample_labeled.csv')]
LABELS = ['pos','neu','neg','q']
print('versions:', platform.python_version(), 'sklearn', sklearn.__version__, 'pandas',pd.__version__, 'pythainlp',pythainlp.__version__)
print('rows:', len(train), len(valid), len(sample))

versions: 3.14.2 sklearn 1.9.1 pandas 3.1.1 pythainlp 5.4.0
rows: 24020 2669 100


## 2.Baseline and controlled experiments
TF-IDF uses character n-grams and default L2 normalization. The ranges are compared with C=1 and no weighting. C and weighting are then compared using the best range by validation macro-F1. The labeled sample is reported but never used to select settings.

In [2]:
def measure(name, vectorizer, model, params):
    start = time.perf_counter()
    X = vectorizer.fit_transform(train.text.fillna(''))
    model.fit(X, train.category)
    fit_seconds = time.perf_counter()-start
    row = {'run':name, **params, 'features':X.shape[1], 'fit_seconds':round(fit_seconds,2)}
    preds = {}
    for split, df in [('valid',valid),('sample',sample)]:
        pred = model.predict(vectorizer.transform(df.text.fillna('')))
        preds[split] = pred
        row[split+'_macro'] = f1_score(df.category,pred,labels=LABELS,average='macro',zero_division=0)
        per = f1_score(df.category,pred,labels=LABELS,average=None,zero_division=0)
        for lab, value in zip(LABELS,per): row[split+'_'+lab] = value
    print(name, 'valid',round(row['valid_macro'],4),'sample',round(row['sample_macro'],4),'features',row['features'],'seconds',row['fit_seconds'],flush=True)
    return row, preds, vectorizer, model
runs=[]
def run_char(name, ngram, C=1.0, weight=None):
    vectorizer=TfidfVectorizer(analyzer='char',ngram_range=ngram,dtype='float32')
    model=LinearSVC(C=C,class_weight=weight,random_state=42,max_iter=10000,dual='auto')
    result=measure(name,vectorizer,model,{'ngram':str(ngram),'C':C,'class_weight':str(weight)})
    runs.append(result)
    return result
baseline=measure('baseline_count_nb',CountVectorizer(tokenizer=lambda s:word_tokenize(str(s),engine='newmm',keep_whitespace=False),token_pattern=None),MultinomialNB(alpha=1.0),{'ngram':'word unigram','C':None,'class_weight':'None'})
for ng in [(2,5),(3,5),(3,6)]: run_char('char_'+str(ng),ng)
range_best=max(runs,key=lambda x:x[0]['valid_macro'])
best_range=eval(range_best[0]['ngram'])
print('Selected range on validation:',best_range)
for c in [0.1,0.5,3.0]: run_char('C_'+str(c),best_range,C=c)
best_c_run=max(runs,key=lambda x:x[0]['valid_macro'])
best_c=best_c_run[0]['C']
run_char('balanced',best_range,C=best_c,weight='balanced')
selected=max(runs,key=lambda x:x[0]['valid_macro'])
print('Selected:',selected[0]['run'])
results=pd.DataFrame([baseline[0]]+[r[0] for r in runs])
print(results.to_string(index=False))

baseline_count_nb valid 0.4458 sample 0.4826 features 29593 seconds 6.51


c:\Users\Acer\my\TSC Project\Thai-sentiment-classification\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


char_(2, 5) valid 0.5587 sample 0.5724 features 1025396 seconds 35.55


c:\Users\Acer\my\TSC Project\Thai-sentiment-classification\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


char_(3, 5) valid 0.5524 sample 0.5635 features 1014154 seconds 26.09


c:\Users\Acer\my\TSC Project\Thai-sentiment-classification\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


... (5 outputs left out)

c:\Users\Acer\my\TSC Project\Thai-sentiment-classification\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


C_3.0 valid 0.5777 sample 0.5611 features 1025396 seconds 44.25


c:\Users\Acer\my\TSC Project\Thai-sentiment-classification\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


balanced valid 0.5848 sample 0.5742 features 1025396 seconds 41.0
Selected: balanced
              run        ngram   C class_weight  features  fit_seconds  valid_macro  valid_pos  valid_neu  valid_neg  valid_q  sample_macro  sample_pos  sample_neu  sample_neg  sample_q
baseline_count_nb word unigram NaN         None     29593         6.51     0.445767   0.440000   0.683966   0.659103 0.000000      0.482581    0.631579    0.651685    0.647059  0.000000
      char_(2, 5)       (2, 5) 1.0         None   1025396        35.55     0.558674   0.443005   0.788786   0.745763 0.257143      0.572435    0.611111    0.754386    0.590909  0.333333
      char_(3, 5)       (3, 5) 1.0         None   1014154        26.09     0.552369   0.445893   0.787376   0.744322 0.231884      0.563464    0.588235    0.741379    0.590909  0.333333
      char_(3, 6)       (3, 6) 1.0         None   1947552        44.71     0.556450   0.443272   0.784753   0.740634 0.257143      0.563464    0.588235    0.741379    0.59

## 3.Error analysis
Review errors from the selected model and compare baseline predictions. Error categories are hypotheses, and validation labels can contain noise.

In [3]:
best_row, best_preds, best_vectorizer, best_model=selected
errors=valid.copy()
errors['baseline_pred']=baseline[1]['valid']; errors['prediction']=best_preds['valid']
errors=errors[errors.category != errors.prediction]
examples=pd.concat([errors[errors.category==label].head(4) for label in LABELS])
print(examples[['id','text','category','prediction','baseline_pred']].to_string(index=False))
print('Selected validation classification report:')
print(classification_report(valid.category,best_preds['valid'],labels=LABELS,zero_division=0))

     id                                                                                                                                                                                                                                                          text category prediction baseline_pred
va00014                                                                                                                                                                                                                                 Vios e ดาวน์/ค่างวด เท่าไรค่ะ      pos        neu           neu
va00027                                                                                                                                                                                                         เคยกินไม่ครบ แต่ต่อไปจะกินสเมอนอฟ เพราะมีสมุนไพร 5555      pos        neu           neg
va00051                                                                                                         

## 4.Optional Export
Set `GROUP_ID` to the real group ID before running this cell. The selected model is trained on train only. The cell validates the output against `sample_submission.csv` and does not send test text externally.

In [4]:
GROUP_ID = 'dingdong'
if GROUP_ID:
    test=pd.read_csv(DATA/'test.csv'); template=pd.read_csv(DATA/'sample_submission.csv')
    submission=pd.DataFrame({'id':test.id,'category':best_model.predict(best_vectorizer.transform(test.text.fillna('')))})
    assert list(submission.columns)==['id','category'] and submission.id.is_unique
    assert submission.id.tolist()==template.id.tolist() and submission.category.isin(LABELS).all()
    output=ROOT/'predictions'/f'{GROUP_ID}_run2_char_tfidf_svm.csv'
    output.parent.mkdir(exist_ok=True); submission.to_csv(output,index=False,encoding='utf-8')
    print('Saved',output,len(submission))
else: print('Submission skipped: set GROUP_ID first.')

Saved c:\Users\Acer\my\TSC Project\Thai-sentiment-classification\predictions\dingdong_run2_char_tfidf_svm.csv 1577
